# Data Visualizations

## Overview
This notebook generates all figures and supporting tables for the research paper (main text + appendix).

**Inputs:** 
- `survey_clean_var.tsv` — cleaned and feature-engineered dataset from `2_data_preparation.ipynb`
- Regression results from `../results/` (generated by `4_modeling.ipynb`)

**Outputs:** Publication-ready figures saved to `../figures/` directory

## Sections
1. **Data Loading & Validation** — Load dataset and verify required columns exist
2. **Main Paper Figures** — Core visualizations for the main text
3. **Appendix Figures** — Supporting visualizations and demographic comparisons
4. **Extra Analysis** — Additional outputs and tables
5. **Statistical Modeling Figures** — Visualizations based on regression results

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import norm
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

In [ ]:
PROJECT_ROOT = Path("..").resolve()
DATA_PATH = PROJECT_ROOT / "survey_clean_var.tsv"
FIGURES_DIR = PROJECT_ROOT / "figures"
FIGURES_DIR.mkdir(exist_ok=True)

In [ ]:
c1 = '#2E86AB'
c2 = '#A23B72'

tri_palette_colors = ['#2E86AB', '#A23B72', '#95a5a6']

blues_palette = ['#1B4F72', '#2E86AB', '#5BA3C5', '#8DC4DC', '#BFE0EE']

sns.set_theme("notebook", style="whitegrid", font_scale=1.3)

# 1. Data Loading & Validation

In [ ]:
df_clean = pd.read_csv(DATA_PATH, sep="\t", encoding="utf-8")
df_clean.head(3)

In [ ]:
# Define all engineered columns expected from 2_data_preparation.ipynb
required_columns = {
    # Demographic groupings
    "EducationGroup", "GeographyGroup", "GenderGroup", "IncomeGroup", "AgeGroup",
    "Eduarea", "Education_STEM",
    # Usage indicators
    "chatbot_user",
    # Language technology experience and literacy
    "LT_exp", "lt_lit", "lt_lit01",
    # GenAI usage intent frequencies
    "InfoRetrieval_freq", "ProblemSolving_freq", "Learning_freq",
    "ContentCreation_freq", "Entertainment_freq", "Creativity_freq",
    # Literacy items (recoded Likert scales)
    "knowledge", "prepared", "limitations", "potential",
    "recognize_errors", "distinguishai", "Education", "Bias_awareness"
}

# Check for missing columns
missing = sorted(required_columns - set(df_clean.columns))
if missing:
    raise ValueError(
        f"Missing {len(missing)} required column(s) in survey_clean_var.tsv:\n"
        f"{', '.join(missing)}\n\n"
        f"These columns should be created by 2_data_preparation.ipynb."
    )

print(f"✓ All {len(required_columns)} required columns present")
print(f"✓ Dataset shape: {df_clean.shape[0]:,} rows × {df_clean.shape[1]} columns")

# 2. Main Paper Figures

## Figure: Language Technology Usage & Replacement by GenAI

This figure shows:
- Panel A: Usage rates of different language technologies
- Panel B: How much GenAI chatbots are replacing each technology

In [ ]:
# ============================================================================
# DATA PREPARATION FOR FIGURE 1: LT USAGE & REPLACEMENT
# ============================================================================
# This cell calculates usage percentages and replacement rates for each 
# language technology. Results include 95% confidence intervals.

# Define mappings to calculate proportions relative the N of users of the specific application
lt_mapping = {
    'Q61': 'Q20-MT',             # MT
    'Q62': 'Q34',                # Speech Transcript
    'Q63': 'Q21',                # Vocal Assistants
    'Q64': 'Q33',                # Assisted Writing
    'Q65': 'Q36',                # Text-to-Speech
    'Q66': 'Q43'                 # Web Search replacement calculated among GenAI users
}

# Labels for what people USE (left side)
usage_labels = {
    'Q61': 'Machine Translation',
    'Q62': 'Speech Transcript',
    'Q63': 'Vocal Assistants',
    'Q64': 'Assisted Writing',
    'Q65': 'Text-to-Speech',
    'Q66': 'GenAI Chatbot'
}

# Labels for what gets REPLACED (right side) 
replacement_labels = {
    'Q61': 'Machine Translation',
    'Q62': 'Speech Transcript',
    'Q63': 'Vocal Assistants',
    'Q64': 'Assisted Writing',
    'Q65': 'Text-to-Speech',
    'Q66': 'Web Search'
}

# Calculate proportion of replacement relative to usage, with CI
# 95% CI z-score
z = norm.ppf(0.975)  
def calc_ci(p, n):
    if n == 0:
        return 0, 0
    margin = z * np.sqrt(p * (1 - p) / n)
    return p * 100, margin * 100

results = []
total = len(df_clean)

for rep_q, use_q in lt_mapping.items():
    if use_q:
        used_mask = ~df_clean[use_q].fillna("").str.contains("Non ho mai usato", case=False)
        
        denominator = used_mask.sum()
        used_pct, used_ci = calc_ci(denominator / total, total)
        rep_series = df_clean.loc[used_mask, rep_q].dropna()
    else:
        denominator = df_clean[rep_q].dropna().count()
        used_pct, used_ci = np.nan, np.nan
        rep_series = df_clean[rep_q].dropna()
    
    comp_count = (rep_series == "Completamente sostituito").sum()
    parz_count = (rep_series == "Parzialmente sostituito").sum()
    comp_pct, comp_ci = calc_ci(comp_count / denominator if denominator else 0, denominator)
    parz_pct, parz_ci = calc_ci(parz_count / denominator if denominator else 0, denominator)
    
    results.append({
        'Technology_Usage': usage_labels[rep_q],
        'Technology_Replacement': replacement_labels[rep_q],
        'Used %': round(used_pct, 1) if not np.isnan(used_pct) else np.nan,
        'Lower Used CI': round(used_pct - used_ci, 1) if not np.isnan(used_pct) else np.nan,
        'Upper Used CI': round(used_pct + used_ci, 1) if not np.isnan(used_pct) else np.nan,
        'Comp %': round(comp_pct, 1),
        'Lower Comp CI': round(comp_pct - comp_ci, 1),
        'Upper Comp CI': round(comp_pct + comp_ci, 1),
        'Parz %': round(parz_pct, 1),
        'Lower Parz CI': round(parz_pct - parz_ci, 1),
        'Upper Parz CI': round(parz_pct + parz_ci, 1),
    })

# Define fixed order for technologies
tech_order = ['Text-to-Speech', 'Assisted Writing', 'Speech Transcript', 'Vocal Assistants', 'Machine Translation', 'GenAI Chatbot']

results_df = pd.DataFrame(results)
# Reorder according to fixed order
results_df['order'] = results_df['Technology_Usage'].map({tech: i for i, tech in enumerate(tech_order)})
results_df = results_df.sort_values('order')

# Data setup
usage_techs = results_df['Technology_Usage'].values
replacement_techs = results_df['Technology_Replacement'].values
n = len(usage_techs)
indices = np.arange(n)

# Extract percentages and calculate error bars
used_pct = results_df['Used %'].values
used_err = [used_pct - results_df['Lower Used CI'].values, 
            results_df['Upper Used CI'].values - used_pct]

comp_pct = results_df['Comp %'].values
comp_err = [comp_pct - results_df['Lower Comp CI'].values,
            results_df['Upper Comp CI'].values - comp_pct]

parz_pct = results_df['Parz %'].values  
parz_err = [parz_pct - results_df['Lower Parz CI'].values,
            results_df['Upper Parz CI'].values - parz_pct]

In [ ]:
# ============================================================================
# PLOT FIGURE 1: DUAL-PANEL USAGE & REPLACEMENT VISUALIZATION
# ============================================================================
# Outputs: usage_shift_breakdown.png, usage_shift_breakdown.pdf

# Create figure with two subplots side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
plt.subplots_adjust(wspace=0.05)

# === Panel A: Usage ===
ax1.grid(True, axis='y', alpha=0.3)
ax1.grid(False, axis='x')
ax1.spines['top'].set_visible(False)
ax1.spines['bottom'].set_visible(False)
ax1.spines['right'].set_visible(False)

ax1.barh(indices, -used_pct, height=0.3, color=tri_palette_colors[2], xerr=used_err, error_kw=dict(ecolor=tri_palette_colors[2], capsize=4), label='Usage')

ax1.set_yticks(indices)
y_labels = [f"{usage}" for usage in usage_techs]
ax1.set_yticklabels(y_labels, ha='right', fontsize=13)
ax1.tick_params(axis='y', pad=15, labelsize=13, length=0, which='both')  
ax1.tick_params(axis='x', labelsize=13)

ax1.set_ylim(-0.5, n-0.5)
ax1.set_title("a) Language Technology Usage", fontsize=14, pad=20, fontweight='bold')
ax1.axvline(0, color='black', linewidth=1.7)

max_used = max(used_pct)
ax1.set_xlim(-max_used * 1.07, 0)
ticks1 = ax1.get_xticks()
ax1.set_xticks(ticks1)  
ax1.set_xticklabels([f'{abs(tick):.0f}' if abs(tick) <= 100 else '' for tick in ticks1])

# Value annotations
for i in range(n):
    ax1.text(-used_pct[i] - max_used*0.05, i, f"{used_pct[i]:.0f}%", va='center', ha='right', color='gray', fontsize=11, weight='bold')

# === Panel B: Replacement ===
ax2.grid(True, axis='y', alpha=0.3)
ax2.grid(False, axis='x')
ax2.spines['top'].set_visible(False)
ax2.spines['bottom'].set_visible(False)
ax2.spines['left'].set_visible(False)

ax2.barh(indices - 0.07, comp_pct, height=0.15, color='#2E86AB', xerr=comp_err, error_kw=dict(ecolor='#2E86AB', capsize=4), 
        label='Completely Replaced')
ax2.barh(indices + 0.07, parz_pct, height=0.15, color=tri_palette_colors[1], xerr=parz_err, error_kw=dict(ecolor=tri_palette_colors[1], capsize=4),
        label='Partially Replaced')

ax2_right = ax2.twinx()
ax2_right.set_yticks(indices)
ax2_right.set_yticklabels(replacement_techs, ha='left', fontsize=13)
ax2_right.tick_params(axis='y', length=0, which='both', pad=15, labelsize=13)  
ax2_right.spines['top'].set_visible(False)
ax2_right.spines['bottom'].set_visible(False)
ax2_right.spines['left'].set_visible(False)
ax2_right.spines['right'].set_visible(False)
ax2_right.grid(False)
ax2_right.set_ylim(-0.5, n-0.5)

ax2.tick_params(axis='x', labelsize=12)
ax2.tick_params(axis='y', length=0, which='both') 
ax2.set_ylim(-0.5, n-0.5)
ax2.set_title("b) Replacement by GenAI Chatbot", fontsize=14, pad=20, fontweight='bold')
ax2.axvline(0, color='black', linewidth=1)

max_repl = max(max(comp_pct), max(parz_pct))
ax2.set_xlim(0, max_repl * 1.35) 
ticks2 = ax2.get_xticks()
ax2.set_xticks(ticks2)  
ax2.set_xticklabels([f'{tick:.0f}' for tick in ticks2])  

# Value annotations
for i in range(n):
    ax2.text(comp_pct[i] + max_repl*0.17, i - 0.15, f"{comp_pct[i]:.0f}%", va='center', ha='left',color='#2E86AB', fontsize=11, weight='bold')
    ax2.text(parz_pct[i] + max_repl*0.17, i + 0.15, f"{parz_pct[i]:.0f}%", va='center', ha='left',color=tri_palette_colors[1], fontsize=11, weight='bold')

# Combined legend below both panels
handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()
fig.legend(handles1 + handles2, labels1 + labels2, loc='lower center', bbox_to_anchor=(0.5, -0.05), ncol=3, framealpha=0.9, fontsize=13)

plt.tight_layout()
plt.savefig('../figures/usage_shift_breakdown.png', bbox_inches='tight')      
plt.savefig('../figures/usage_shift_breakdown.pdf', bbox_inches='tight')      
plt.show()

## Figure: GenAI Chatbot Usage Patterns

Four-panel figure analyzing GenAI usage:
- Panel A: Usage intent frequency distribution
- Panel B: Activities breakdown (work vs personal)
- Panel C: Distribution of activities per user
- Panel D: Work/study usage by profession

In [ ]:
# ============================================================================
# DATA PREPARATION FOR FIGURE 2: GENAI USAGE PATTERNS
# ============================================================================
# Define labels, mappings, and helper structures for the multi-panel figure

# GenAI intent columns and labels
cols_genai = ['Q51_1', 'Q51_2', 'Q51_3', 'Q51_4', 'Q51_5', 'Q51_6']
labels_genai_it = ["Recupero info", "Risoluzione problemi", "Apprendimento", "Creazione contenuti", "Intrattenimento", "Creatività"]
freq_order_it = ["Almeno una volta a settimana", "Almeno una volta al mese", "Meno di una volta al mese", "Mai"]

# English translations
labels_genai_en = ["Information\nRetrieval", "Problem\nSolving", "Learning", "Content\nCreation", "Leisure", "Creativity"]
freq_order_en = ["Weekly", "Monthly", "< Monthly", "Never"]
purpose_mapping = dict(zip(labels_genai_it, labels_genai_en))
freq_mapping = dict(zip(freq_order_it, freq_order_en))

# Scale mapping for average calculation
freq_scale = {"Never": 0, "< Monthly": 1, "Monthly": 2, "Weekly": 3}

# Activity labels (shortened for better display)
q53_to_label_en = {
    "Q53_1": "Email Writing", "Q53_2": "Creative Writing", "Q53_3": "Academic Writing",
    "Q53_4": "Other Writing", "Q53_5": "Text Analysis", "Q53_6": "Idea Generation",
    "Q53_7": "Explanations", "Q53_8": "Summaries", "Q53_9": "Quiz Creation",
    "Q53_10": "Travel Planning", "Q53_11": "Fact Checking", "Q53_12": "Personal Advice",
    "Q53_13": "Medical Advice", "Q53_14": "Other Advice", "Q53_15": "Emotional Support",
    "Q53_16": "Casual Chat", "Q53_17": "Romantic Chat", "Q53_18": "Philosophical Chat",
    "Q53_19": "Other Chat", "Q53_20": "Programming Help", "Q53_21": "Data Analysis",
    "Q53_22": "Image Generation", "Q53_23": "Audio/Music Gen.", "Q53_25": "Other"
}

response_types_it = ['Studio/Lavoro', 'Svago/Uso personale']
response_types_en = ['Work/Study', 'Personal/Leisure']

# Extended mapping for all occupations to English
extended_occ_mapping = {
    'In pensione': 'Retired',
    'IT e media': 'IT & Media', 
    'Formazione': 'Education',
    'Ricerca e accademia': 'Research & Academia',
    'Impresa e consulenza aziendale': 'Business & Consulting',
    'Sicurezza e pubblica amministrazione': 'Security & Public Admin',
    'Studente': 'Student',
    'Industria e trasporti': 'Industry & Transport',
    'Sanità': 'Healthcare',
    'Non lavoro al momento': 'Currently Not Working',
    'Finanza': 'Finance',
    'Cultura e spettacolo': 'Culture & Entertainment',
    'Turismo e ristorazione': 'Services & Hospitality',
    'Costruzioni': 'Construction',
    'Agricoltura e ambiente': 'Agriculture & Environment'
}

In [ ]:
# Helper function for binomial confidence intervals
def binomial_ci(count, n, confidence=0.95):
    """Calculate binomial proportion confidence interval."""
    if n == 0:
        return 0, 0, 0
    p = count / n
    alpha = 1 - confidence
    z = stats.norm.ppf(1 - alpha/2)
    margin = z * np.sqrt(p * (1 - p) / n)
    return p * 100, max(0, (p - margin) * 100), min(100, (p + margin) * 100)

In [ ]:
# ============================================================================
# PLOT FIGURE 2: GENAI USAGE PATTERNS (4-PANEL)
# ============================================================================
# Outputs: genai_usage_patterns.png, genai_usage_patterns.pdf
#
# Panel A: Usage intent frequency distribution with average scores
# Panel B: Activities by work/study vs personal usage
# Panel C: Distribution of number of activities per user
# Panel D: Work/study usage percentage by profession (with sample sizes)

fig = plt.figure(figsize=(28, 24))
gs = fig.add_gridspec(2, 2, height_ratios=[1, 1], width_ratios=[1, 1], 
                      hspace=0.35, wspace=0.55, 
                      left=0.2, right=0.95, top=0.91, bottom=0.07)

# PANEL A: Usage Intent (Top Left)
ax_a = fig.add_subplot(gs[0, 0])

# Process frequency data
df_long = df_clean.loc[:, cols_genai].melt(var_name='Purpose', value_name='Frequency')
df_long['Purpose'] = df_long['Purpose'].map(dict(zip(cols_genai, labels_genai_it)))
df_long['Purpose'] = df_long['Purpose'].map(purpose_mapping)
df_long['Frequency'] = df_long['Frequency'].map(freq_mapping)
df_long['Frequency_Score'] = df_long['Frequency'].map(freq_scale)

# Calculate frequency distributions and statistics with CI
freq_stats = []
for purpose in labels_genai_en:
    purpose_data = df_long[df_long['Purpose'] == purpose]
    freq_dist = purpose_data['Frequency'].value_counts()
    total = freq_dist.sum()
    
    # Calculate percentages with CI
    weekly_count = freq_dist.get('Weekly', 0)
    monthly_count = freq_dist.get('Monthly', 0)
    less_monthly_count = freq_dist.get('< Monthly', 0)
    
    weekly_pct, weekly_ci_low, weekly_ci_high = binomial_ci(weekly_count, total)
    monthly_pct, monthly_ci_low, monthly_ci_high = binomial_ci(monthly_count, total)
    less_monthly_pct, less_monthly_ci_low, less_monthly_ci_high = binomial_ci(less_monthly_count, total)
    
    # Calculate mean and CI for scores
    scores = purpose_data['Frequency_Score'].dropna()
    mean_score = scores.mean()
    if len(scores) > 1:
        sem = stats.sem(scores)
        ci_lower, ci_upper = stats.t.interval(0.95, len(scores)-1, loc=mean_score, scale=sem)
    else:
        ci_lower, ci_upper = mean_score, mean_score
    
    freq_stats.append({
        'Purpose': purpose,
        'Weekly': weekly_pct,
        'Weekly_CI_Low': weekly_ci_low,
        'Weekly_CI_High': weekly_ci_high,
        'Monthly': monthly_pct,
        'Less_Monthly': less_monthly_pct,
        'Mean_Score': mean_score,
        'CI_Lower': ci_lower,
        'CI_Upper': ci_upper,
        'Sample_Size': total
    })

freq_df = pd.DataFrame(freq_stats).sort_values('Mean_Score', ascending=True)

# Create stacked horizontal bar with average overlay
y_pos = np.arange(len(freq_df))
colors = ['#2E86AB', '#A23B72', '#95a5a6']

# Stacked bars
bars1 = ax_a.barh(y_pos, freq_df['Weekly'], label='Weekly', color=colors[0], alpha=0.8)
bars2 = ax_a.barh(y_pos, freq_df['Monthly'], left=freq_df['Weekly'], label='Monthly', color=colors[1], alpha=0.8)
bars3 = ax_a.barh(y_pos, freq_df['Less_Monthly'], left=freq_df['Weekly']+freq_df['Monthly'], 
                label='< Monthly', color=colors[2], alpha=0.8)

# Add average frequency scores
total_active = freq_df['Weekly'] + freq_df['Monthly'] + freq_df['Less_Monthly']
for i, (total_pct, mean_score) in enumerate(zip(total_active, freq_df['Mean_Score'])):
    ax_a.text(total_pct + 2, i, f'{mean_score:.2f}', 
            ha='left', va='center', fontweight='bold', fontsize=24, 
            bbox=dict(boxstyle="round,pad=0.3", facecolor='white', alpha=0.7, edgecolor='gray', linewidth=0.5))

ax_a.set_yticks(y_pos)
ax_a.set_yticklabels(freq_df['Purpose'], fontsize=24)
ax_a.set_xlabel('Users (Weekly + Monthly + < Monthly) %', fontsize=24, fontweight='bold')
ax_a.set_title('a) Usage Intent: Frequency Distribution & Average', fontsize=28, fontweight='bold', pad=25)
ax_a.legend(loc='lower right', fontsize=24)
ax_a.set_xlim(0, 110)
ax_a.tick_params(axis='both', which='major', labelsize=24)

# PANEL B: Activities with Work/Personal Breakdown (Top Right)
ax_b = fig.add_subplot(gs[0, 1])

# Calculate activity data
q53_columns = [col for col in q53_to_label_en.keys() if col in df_clean.columns]
activity_data = {"activity": [], "Work/Study": [], "Personal/Leisure": [], "Work_CI_Low": [], 
                "Work_CI_High": [], "Personal_CI_Low": [], "Personal_CI_High": []}

for col in q53_columns:
    activity_label_en = q53_to_label_en.get(col, col)
    responses = df_clean[col].dropna().astype(str)
    total_responses = len(responses)
    
    count_work = sum(response_types_it[0] in resp for resp in responses)
    count_personal = sum(response_types_it[1] in resp for resp in responses)
    
    # Calculate CI
    if total_responses > 0:
        work_pct, work_ci_low, work_ci_high = binomial_ci(count_work, total_responses)
        personal_pct, personal_ci_low, personal_ci_high = binomial_ci(count_personal, total_responses)
    else:
        work_pct = work_ci_low = work_ci_high = 0
        personal_pct = personal_ci_low = personal_ci_high = 0
    
    activity_data['activity'].append(activity_label_en)
    activity_data['Work/Study'].append(count_work)
    activity_data['Personal/Leisure'].append(count_personal)
    activity_data['Work_CI_Low'].append(work_ci_low)
    activity_data['Work_CI_High'].append(work_ci_high)
    activity_data['Personal_CI_Low'].append(personal_ci_low)
    activity_data['Personal_CI_High'].append(personal_ci_high)

activity_df = pd.DataFrame(activity_data)
activity_df['Total'] = activity_df['Work/Study'] + activity_df['Personal/Leisure']
activity_df = activity_df.sort_values('Total', ascending=True)

y_pos = np.arange(len(activity_df))
bars1 = ax_b.barh(y_pos, activity_df['Work/Study'], label='Work/Study', color=colors[0], alpha=0.8)
bars2 = ax_b.barh(y_pos, activity_df['Personal/Leisure'], left=activity_df['Work/Study'], label='Personal', color=colors[2], alpha=0.8)

ax_b.set_yticks(y_pos)
ax_b.set_yticklabels(activity_df['activity'], fontsize=24)
ax_b.set_xlabel('Total Mentions of each Activity', fontsize=24, fontweight='bold')
ax_b.set_title('b) Activities: Work/Study vs Personal Usage', fontsize=28, fontweight='bold', pad=25)
ax_b.legend(loc='lower right', fontsize=24)
ax_b.tick_params(axis='both', which='major', labelsize=24)

# PANEL C: Distribution of Activities per User (Bottom Left)
ax_c = fig.add_subplot(gs[1, 0])

if 'Q52' in df_clean.columns:
    activity_counts = (
        df_clean['Q52']
        .dropna()
        .str.split(',')
        .apply(lambda x: len([i.strip() for i in x if i.strip() not in ['Altro', 'Nessuna di queste']]))
    )
    
    hist_data = activity_counts.value_counts().sort_index()
    
    start_color = '#8DC4DC' 
    end_color = '#1B4F72'
    custom_cmap = LinearSegmentedColormap.from_list("custom_blue", [start_color, end_color])
    n_colors = len(hist_data)
    colors_engagement = custom_cmap(np.linspace(0, 1, n_colors))

    bars = ax_c.bar(hist_data.index, hist_data.values, color=colors_engagement)
    
    total_users = hist_data.sum()
    
    # Calculate and plot AVERAGE line
    avg_activities = activity_counts.mean()
    ax_c.axvline(x=avg_activities, color=colors[0], 
                 linestyle='--', alpha=0.9, linewidth=3)
    ax_c.text(avg_activities + 0.3, max(hist_data.values)*0.95, 
          f'Avg: {avg_activities:.2f}', 
            ha='left', fontsize=24, fontweight='bold',
            bbox=dict(boxstyle="round,pad=0.4", facecolor='white', alpha=0.7, edgecolor='gray', linewidth=0.5))

ax_c.set_xlabel('Number of Activities', fontsize=24, fontweight='bold')
ax_c.set_ylabel('Number of Users', fontsize=24, fontweight='bold')
ax_c.set_title('c) Distribution of Activities per User', fontsize=28, fontweight='bold', pad=30)
ax_c.tick_params(axis='both', which='major', labelsize=24)
ax_c.grid(True, alpha=0.2, linestyle='--', axis='y')

# PANEL D: Work/Study Usage % by Occupation (Bottom Right)
ax_d = fig.add_subplot(gs[1, 1])

occupation_work_data = []

if 'mapped_job' in df_clean.columns and 'chatbot_user' in df_clean.columns:
    df_chatbot_users = df_clean[df_clean['chatbot_user'] == 'Yes']
    
    all_occupations = df_chatbot_users['mapped_job'].dropna().unique()
    all_occupations = [occ for occ in all_occupations if occ not in ['Altro', 'Agricoltura e ambiente', "Costruzioni"]]
    
    for occ_it in all_occupations:
        occ_subset = df_chatbot_users[df_chatbot_users['mapped_job'] == occ_it]
        
        if len(occ_subset) > 0:
            total_work = 0
            total_personal = 0
            
            for col in q53_columns:
                if col in occ_subset.columns:
                    responses = occ_subset[col].dropna().astype(str)
                    total_work += sum(response_types_it[0] in resp for resp in responses)
                    total_personal += sum(response_types_it[1] in resp for resp in responses)
            
            if total_work + total_personal > 0:
                total_usage = total_work + total_personal
                work_pct = total_work / total_usage * 100
                
                _, ci_low, ci_high = binomial_ci(total_work, total_usage)
                occ_en = extended_occ_mapping.get(occ_it, occ_it)
                
                occupation_work_data.append({
                    'Occupation': occ_en,
                    'Work_Pct': work_pct,
                    'CI_Low': ci_low,
                    'CI_High': ci_high,
                    'Sample_Size': len(occ_subset) 
                })
                
if occupation_work_data:
    work_df = pd.DataFrame(occupation_work_data).sort_values('Work_Pct', ascending=True)
    
    norm = plt.Normalize(vmin=work_df['Work_Pct'].min(), vmax=work_df['Work_Pct'].max())
    start_color = '#BFE0EE' 
    end_color = '#1B4F72'
    custom_cmap = LinearSegmentedColormap.from_list("custom_blue", [start_color, end_color])
    n_colors = len(work_df)
    colors_lollipop = custom_cmap(norm(work_df['Work_Pct']))

    y_pos = np.arange(len(work_df))
    
    # Draw lollipop sticks
    for i, (y, x, n) in enumerate(zip(y_pos, work_df['Work_Pct'], work_df['Sample_Size'])):
        stick_width = 1 + (n / work_df['Sample_Size'].max()) * 2
        ax_d.plot([0, x], [y, y], color='gray', alpha=0.5, linewidth=stick_width, zorder=1)
    
    # Draw CI error bars
    xerr_low = work_df['Work_Pct'] - work_df['CI_Low']
    xerr_high = work_df['CI_High'] - work_df['Work_Pct']
    ax_d.errorbar(work_df['Work_Pct'], y_pos, 
                xerr=[xerr_low, xerr_high],
                fmt='none', ecolor='black', alpha=0.4, 
                capsize=5, capthick=2, linewidth=1.5, zorder=2)
    
    # Draw lollipop heads
    scatter = ax_d.scatter(work_df['Work_Pct'], y_pos, 
                         s=200,
                         c=colors_lollipop, 
                         alpha=0.85, 
                         edgecolor='black', 
                         linewidth=2.5, 
                         zorder=3)
    
    # Add sample size labels
    for i, (work_pct, sample_size) in enumerate(zip(work_df['Work_Pct'], work_df['Sample_Size'])):
        ax_d.text(work_pct + 2.5, i, f'n={sample_size}', 
                va='center', ha='left', fontsize=24, fontweight='bold',
                bbox=dict(boxstyle="round,pad=0.25", facecolor='white', alpha=0.7, edgecolor='gray', linewidth=0.5))
    
    ax_d.set_yticks(y_pos)
    ax_d.set_yticklabels(work_df['Occupation'], fontsize=24)
    ax_d.set_xlabel('Work/Study Usage (% of total usage) [95% CI]', fontsize=24, fontweight='bold')
    ax_d.set_title('d) Work/Study Usage % by Profession', fontsize=28, fontweight='bold', pad=30)
    ax_d.set_xlim(-3, 115)
    ax_d.grid(True, alpha=0.25, axis='x', linestyle='--', linewidth=0.8)
    ax_d.axvline(x=50, color=colors[1], linestyle='--', linewidth=2.5, zorder=0)
    ax_d.text(50, len(work_df)-0.5, '50%', ha='center', fontsize=24, fontweight='bold',
            bbox=dict(boxstyle="round,pad=0.3", facecolor='white', alpha=0.7, edgecolor='gray', linewidth=0.5))
    ax_d.tick_params(axis='both', which='major', labelsize=24)

# Update rcParams for consistent styling
plt.rcParams.update({
    "axes.titlesize": 28,
    "axes.titleweight": "bold",
    "axes.labelsize": 24,
    "axes.labelweight": "bold",
    "xtick.labelsize": 24,
    "ytick.labelsize": 24,
    "legend.fontsize": 24,
})

plt.suptitle('GenAI Chatbots Usage Patterns', fontsize=34, fontweight='bold', y=1.0)
plt.savefig('../figures/genai_usage_patterns.png', dpi=300, bbox_inches='tight')
plt.savefig('../figures/genai_usage_patterns.pdf', dpi=300, bbox_inches='tight')

plt.show()

## Figure: Activity Usage by Age Group

Visualizes how different age groups engage with specific GenAI activities.

In [ ]:
# ============================================================================
# PLOT FIGURE 3: ACTIVITY USAGE BY AGE GROUP
# ============================================================================
# Outputs: activities_by_age_selected_ci.png, activities_by_age_selected_ci.pdf
#
# Visualizes how different age groups engage with specific GenAI activities,
# with 95% confidence intervals for each age group.

# Select specific activities
selected_activities = {
    "Q53_5": "Text Analysis",
    "Q53_7": "Explanations",
    "Q53_11": "Fact Checking",
    "Q53_12": "Personal Advice",
    "Q53_13": "Medical Advice",
    "Q53_15": "Emotional Support",
    "Q53_20": "Programming Help",
    "Q53_21": "Data Analysis"
}

# Filter for chatbot users only
df_chatbot = df_clean[df_clean['chatbot_user'] == 'Yes']

# Get activity counts and sort
activity_counts = []
for col, label in selected_activities.items():
    if col in df_chatbot.columns:
        count = df_chatbot[col].notna().sum()
        activity_counts.append((label, count, col))

# Sort by total usage
activity_counts_sorted = sorted(activity_counts, key=lambda x: x[1], reverse=True)
activities_sorted = [act[0] for act in activity_counts_sorted]
columns_sorted = [act[2] for act in activity_counts_sorted]

# Create custom blue palette
start_color = '#BFE0EE'
end_color = '#1B4F72'
custom_cmap = LinearSegmentedColormap.from_list("custom_blue", [start_color, end_color])

# Create figure
fig, ax = plt.subplots(1, 1, figsize=(12, 6))

if 'AgeGroup' in df_chatbot.columns:
    age_groups = sorted(df_chatbot['AgeGroup'].dropna().unique())
    
    # Calculate percentages with CI
    age_data = {}
    age_ci_lower = {}
    age_ci_upper = {}
    age_sizes = {}
    
    for age in age_groups:
        age_subset = df_chatbot[df_chatbot['AgeGroup'] == age]
        age_sizes[age] = len(age_subset)
        age_data[age] = []
        age_ci_lower[age] = []
        age_ci_upper[age] = []
        
        for col in columns_sorted:
            if col in age_subset.columns:
                count = age_subset[col].notna().sum()
                pct, ci_low, ci_high = binomial_ci(count, age_sizes[age])
            else:
                pct, ci_low, ci_high = 0, 0, 0
            
            age_data[age].append(pct)
            age_ci_lower[age].append(ci_low)
            age_ci_upper[age].append(ci_high)
    
    # Plot vertical bars with error bars
    x = np.arange(len(activities_sorted))
    width = 0.8 / len(age_groups)
    
    colors_age = custom_cmap(np.linspace(0, 1, len(age_groups)))
    
    for i, age in enumerate(age_groups):
        offset = (i - len(age_groups)/2 + 0.5) * width
        x_positions = x + offset
        
        # Plot bars
        bars = ax.bar(x_positions, age_data[age], width, 
                     label=f'{age} (n={age_sizes[age]})', 
                     alpha=0.85, color=colors_age[i])
        
        # Calculate error bar values
        yerr_lower = np.array(age_data[age]) - np.array(age_ci_lower[age])
        yerr_upper = np.array(age_ci_upper[age]) - np.array(age_data[age])
        
        # Add error bars
        ax.errorbar(x_positions, age_data[age], 
                   yerr=[yerr_lower, yerr_upper],
                   fmt='none', ecolor='black', alpha=0.5, 
                   capsize=3, capthick=1.5, linewidth=1.5, zorder=10)
    
    ax.set_ylabel('% of Age Group performing Activity')
    ax.set_title('Activity Usage by Age Group', pad=20)
    ax.set_xticks(x)
    ax.set_xticklabels(activities_sorted, rotation=45, ha='right')
    ax.legend(loc='upper right', framealpha=0.9)
    ax.grid(True, alpha=0.3, axis='y')
    
    # Set y-axis limit to accommodate error bars
    max_val = max([max(age_ci_upper[age]) for age in age_groups])
    ax.set_ylim(0, max_val * 1.1)

plt.tight_layout()
plt.savefig('../figures/activities_by_age_selected_ci.png', dpi=300, bbox_inches='tight')
plt.savefig('../figures/activities_by_age_selected_ci.pdf', dpi=300, bbox_inches='tight')
plt.show()

## Figure: Language Technology Literacy & Attitudes

Three-panel comparison of GenAI users vs non-users:
- Panel A: LT literacy items (knowledge, preparedness, limitations, etc.)
- Panel B: Attitude toward LT education
- Panel C: Bias awareness

In [ ]:
# Define all Likert items 
likert_items = {
    'likert_1_1': 'knowledge',           # I have good knowledge of LT
    'likert_1_3': 'prepared',            # I feel prepared to use LT
    'likert_1_4': 'limitations',         # I know the limitations of LT
    'likert_1_5': 'potential',           # I know the potential of LT
    'likert_2_2': 'recognize_errors',    # I can recognize errors in LT
    'likert_2_3': 'distinguishai',       # I can distinguish AI text from human text
    'likert_1_6': 'Education',           # Education should be part of programs
    'likert_2_1': 'Bias_awareness'       # Aware that LT can reproduce biases
}

In [ ]:
# Recoding function: center at "Non so" (2)
def recode_likert_centered(series):
    """
    Recode Likert scale so that 'Non so' (2) becomes 0
    0 -> -2 (strongly disagree)
    1 -> -1 (disagree)
    2 -> 0  (don't know/neutral)
    3 -> 1  (agree)
    4 -> 2  (strongly agree)
    """
    return series - 2

# Recode the Likert items and create new columns with short names
for original_col, short_name in likert_items.items():
    if original_col in df_clean.columns:
        df_clean[short_name] = recode_likert_centered(df_clean[original_col])

In [ ]:
def calculate_stats(df, column):
    """Calculate mean and 95% CI for a column"""
    data = df[column].dropna()
    n = len(data)
    mean = data.mean()
    se = data.std() / np.sqrt(n)
    ci = 1.96 * se
    return mean, mean - ci, mean + ci, n

In [ ]:
# ============================================================================
# PLOT FIGURE 4: LT LITERACY & ATTITUDES (3-PANEL)
# ============================================================================
# Outputs: lt_attitudes_literacy.png, lt_attitudes_literacy.pdf
#
# Compares GenAI users vs non-users across:
# - Panel A: Literacy items (knowledge, preparedness, limitations, etc.)
# - Panel B: Attitude toward LT education in programs
# - Panel C: Awareness that LT can reproduce biases

# Filter data
df_users = df_clean[df_clean['chatbot_user'] == 'Yes'].copy()
df_nonusers = df_clean[df_clean['chatbot_user'] == 'No'].copy()
df_all = df_clean[df_clean['GenderGroup'].isin(['Man', 'Woman'])].copy()

# Define items for literacy plot
literacy_items = {
    'knowledge': 'Good knowledge of LT',
    'prepared': 'Feel prepared to use LT',
    'limitations': 'Know limitations of LT',
    'potential': 'Know potential of LT',
    'recognize_errors': 'Can recognize errors in LT',
    'distinguishai': 'Can distinguish AI from human text'
}

# Font sizes
TITLE_FONTSIZE = 20
LABEL_FONTSIZE = 16
TICK_FONTSIZE = 18
ANNOT_FONTSIZE = 16
LEGEND_FONTSIZE = 18

# Create figure
fig = plt.figure(figsize=(16, 8.5))
gs = fig.add_gridspec(2, 2, width_ratios=[2, 1], height_ratios=[1, 1],
                      hspace=0.35, wspace=0.3, bottom=0.12)

# --- Panel A: Literacy items ---
ax_lit = fig.add_subplot(gs[:, 0])
items = list(literacy_items.keys())
labels = [literacy_items[item] for item in items]
y_pos = np.arange(len(items))
user_means = []
nonuser_means = []
all_means = []
p_values = []

for item in items:
    u_mean, _, _, _ = calculate_stats(df_users, item)
    nu_mean, _, _, _ = calculate_stats(df_nonusers, item)
    a_mean, _, _, _ = calculate_stats(df_all, item)
    
    users_data = df_users[item].dropna()
    nonusers_data = df_nonusers[item].dropna()
    
    try:
        _, p_val = stats.ttest_ind(users_data, nonusers_data, equal_var=False, nan_policy='omit')
    except Exception:
        p_val = np.nan
    
    user_means.append(u_mean)
    nonuser_means.append(nu_mean)
    all_means.append(a_mean)
    p_values.append(p_val)

# Plot connecting lines
for i in range(len(items)):
    ax_lit.plot([nonuser_means[i], user_means[i]], [i, i],
                color='gray', linewidth=1.5, alpha=0.4, zorder=1)

# Plot dots
for i in range(len(items)):
    is_sig = (not np.isnan(p_values[i])) and (p_values[i] < 0.05)
    user_color = '#2E86AB' if is_sig else '#66B2C1'
    nonuser_color = '#A23B72' if is_sig else '#C77B9E'
    
    ax_lit.scatter(user_means[i], i, s=180, color=user_color,
                   label='GenAI Users' if i == 0 else '', zorder=3,
                   edgecolors='white', linewidth=2)
    ax_lit.scatter(nonuser_means[i], i, s=180, color=nonuser_color,
                   label='Non-Users' if i == 0 else '', zorder=3,
                   edgecolors='white', linewidth=2)

# 'All' means
ax_lit.scatter(all_means, y_pos, s=180, color='#95a5a6', marker='D',
               label='All', zorder=3, edgecolors='white', linewidth=2)

# Add value labels
for i, (u_val, nu_val, a_val) in enumerate(zip(user_means, nonuser_means, all_means)):
    is_sig = (not np.isnan(p_values[i])) and (p_values[i] < 0.05)
    user_color = '#2E86AB' if is_sig else '#66B2C1'
    nonuser_color = '#A23B72' if is_sig else '#C77B9E'
    
    ax_lit.text(u_val + 0.08, i + 0.12, f'{u_val:.2f}', va='center',
                fontsize=ANNOT_FONTSIZE, color=user_color, fontweight='bold')
    ax_lit.text(nu_val - 0.08, i + 0.12, f'{nu_val:.2f}', va='center', ha='right',
                fontsize=ANNOT_FONTSIZE, color=nonuser_color, fontweight='bold')
    ax_lit.text(a_val, i - 0.18, f'{a_val:.2f}', va='top', ha='center',
                fontsize=ANNOT_FONTSIZE, color='#95a5a6', fontweight='bold')

ax_lit.axvline(x=0, color='black', linewidth=2, linestyle='--', alpha=0.3)
ax_lit.set_yticks(y_pos)
ax_lit.set_yticklabels(labels, fontsize=TICK_FONTSIZE)
ax_lit.set_xlabel('Mean (0=neutral)', fontsize=LABEL_FONTSIZE, fontweight='bold')
ax_lit.set_title('a) LT Literacy: GenAI Users vs Non-Users', fontsize=TITLE_FONTSIZE,
                 fontweight='bold', pad=30)
ax_lit.set_xlim(-1.6, 1.6)
ax_lit.set_xticks([-1.5, -1, -0.5, 0, 0.5, 1, 1.5])
ax_lit.set_ylim(-0.5, len(items) - 0.5)
ax_lit.grid(axis='x', alpha=0.2, linestyle=':')
ax_lit.spines['top'].set_visible(False)
ax_lit.spines['right'].set_visible(False)
ax_lit.tick_params(axis='x', labelsize=TICK_FONTSIZE)

# --- Panel B: Education ---
ax_edu = fig.add_subplot(gs[0, 1])

item = 'Education'
u_mean, u_lower, u_upper, _ = calculate_stats(df_users, item)
nu_mean, nu_lower, nu_upper, _ = calculate_stats(df_nonusers, item)
a_mean, a_lower, a_upper, _ = calculate_stats(df_all, item)

users_data = df_users[item].dropna()
nonusers_data = df_nonusers[item].dropna()
try:
    _, p_val = stats.ttest_ind(users_data, nonusers_data, equal_var=False, nan_policy='omit')
except Exception:
    p_val = np.nan
is_sig = (not np.isnan(p_val)) and (p_val < 0.05)

user_color = '#2E86AB' if is_sig else '#66B2C1'
nonuser_color = '#A23B72' if is_sig else '#C77B9E'

ax_edu.plot([0, 0], [nu_mean, u_mean], color='gray', linewidth=1.5, alpha=0.4, zorder=2)
ax_edu.scatter(0, u_mean, s=180, color=user_color, zorder=3, edgecolors='white', linewidth=2)
ax_edu.scatter(0, nu_mean, s=180, color=nonuser_color, zorder=3, edgecolors='white', linewidth=2)
ax_edu.scatter(0, a_mean, s=180, color='#95a5a6', marker='D', zorder=3, edgecolors='white', linewidth=2)

ax_edu.text(0.1, u_mean, f'{u_mean:.2f}', va='center', fontsize=ANNOT_FONTSIZE, color=user_color, fontweight='bold')
ax_edu.text(0.1, nu_mean, f'{nu_mean:.2f}', va='center', fontsize=ANNOT_FONTSIZE, color=nonuser_color, fontweight='bold')
ax_edu.text(-0.1, a_mean, f'{a_mean:.2f}', va='center', ha='right', fontsize=ANNOT_FONTSIZE, color='#95a5a6', fontweight='bold')

ax_edu.axhline(y=0, color='black', linewidth=2, linestyle='--', alpha=0.3)
ax_edu.set_xticks([0])
ax_edu.set_xticklabels([''], fontsize=TICK_FONTSIZE)
ax_edu.set_ylabel('Mean', fontsize=LABEL_FONTSIZE, fontweight='bold')
ax_edu.set_title('b) LT Education Attitude', fontsize=TITLE_FONTSIZE, fontweight='bold', pad=30)
ax_edu.set_xlim(-0.5, 0.5)
ax_edu.set_ylim(-1.5, 1.5)
ax_edu.set_yticks([-1.5, -1, -0.5, 0, 0.5, 1, 1.5])
ax_edu.grid(axis='y', alpha=0.2, linestyle=':')
ax_edu.spines['top'].set_visible(False)
ax_edu.spines['right'].set_visible(False)
ax_edu.tick_params(axis='y', labelsize=TICK_FONTSIZE)

# --- Panel C: Bias awareness ---
ax_bias = fig.add_subplot(gs[1, 1])

item = 'Bias_awareness'
u_mean, u_lower, u_upper, _ = calculate_stats(df_users, item)
nu_mean, nu_lower, nu_upper, _ = calculate_stats(df_nonusers, item)
a_mean, a_lower, a_upper, _ = calculate_stats(df_all, item)

users_data = df_users[item].dropna()
nonusers_data = df_nonusers[item].dropna()
try:
    _, p_val = stats.ttest_ind(users_data, nonusers_data, equal_var=False, nan_policy='omit')
except Exception:
    p_val = np.nan
is_sig = (not np.isnan(p_val)) and (p_val < 0.05)

user_color = '#2E86AB' if is_sig else '#66B2C1'
nonuser_color = '#A23B72' if is_sig else '#C77B9E'

# Plot CIs
ax_bias.plot([0, 0], [u_lower, u_upper], color=user_color, linewidth=2, alpha=0.3, zorder=1)
ax_bias.plot([0, 0], [nu_lower, nu_upper], color=nonuser_color, linewidth=2, alpha=0.3, zorder=1)
ax_bias.plot([0, 0], [a_lower, a_upper], color='#95a5a6', linewidth=2, alpha=0.3, zorder=1)

ax_bias.plot([0, 0], [nu_mean, u_mean], color='gray', linewidth=1.5, alpha=0.4, zorder=2)
ax_bias.scatter(0, u_mean, s=180, color=user_color, zorder=3, edgecolors='white', linewidth=2)
ax_bias.scatter(0, nu_mean, s=180, color=nonuser_color, zorder=3, edgecolors='white', linewidth=2)
ax_bias.scatter(0, a_mean, s=180, color='#95a5a6', marker='D', zorder=3, edgecolors='white', linewidth=2)

ax_bias.text(0.1, u_mean, f'{u_mean:.2f}', va='center', fontsize=ANNOT_FONTSIZE, color=user_color, fontweight='bold')
ax_bias.text(0.1, nu_mean, f'{nu_mean:.2f}', va='center', fontsize=ANNOT_FONTSIZE, color=nonuser_color, fontweight='bold')
ax_bias.text(-0.1, a_mean, f'{a_mean:.2f}', va='center', ha='right', fontsize=ANNOT_FONTSIZE, color='#95a5a6', fontweight='bold')

ax_bias.axhline(y=0, color='black', linewidth=2, linestyle='--', alpha=0.3)
ax_bias.set_xticks([0])
ax_bias.set_xticklabels([''], fontsize=TICK_FONTSIZE)
ax_bias.set_ylabel('Mean', fontsize=LABEL_FONTSIZE, fontweight='bold')
ax_bias.set_title('c) Bias Awareness', fontsize=TITLE_FONTSIZE, fontweight='bold', pad=25)
ax_bias.set_xlim(-0.5, 0.5)
ax_bias.set_ylim(-1.5, 1.5)
ax_bias.set_yticks([-1.5, -1, -0.5, 0, 0.5, 1, 1.5])
ax_bias.grid(axis='y', alpha=0.2, linestyle=':')
ax_bias.spines['top'].set_visible(False)
ax_bias.spines['right'].set_visible(False)
ax_bias.tick_params(axis='y', labelsize=TICK_FONTSIZE)

# Add legend
handles, labels = ax_lit.get_legend_handles_labels()
fig.legend(handles, labels, loc='lower right', shadow=True, ncol=3, fontsize=LEGEND_FONTSIZE, 
           framealpha=0.95, bbox_to_anchor=(0.57, -0.06))

plt.savefig('../figures/lt_attitudes_literacy.png', dpi=300, bbox_inches='tight')
plt.savefig('../figures/lt_attitudes_literacy.pdf', dpi=300, bbox_inches='tight')
plt.show()

# 3. Appendix Figures

## Appendix: Sample Demographics vs National Statistics

Compares survey sample demographics to Italian national statistics (ISTAT 2025).

In [ ]:
# Representative distribution

ideal_gender = pd.DataFrame.from_dict({
    'Male': 48.4,
    'Female': 51.6
}, orient='index', columns=['Percentage']).reset_index(names='Category')

ideal_age = pd.DataFrame.from_dict({
    '18–24': 8.3,
    '25–34': 12.5,
    '35–44': 14.1,
    '45–54': 18.3,
    '55–64': 18.2,
    '65+': 28.7
}, orient='index', columns=['Percentage']).reset_index(names='Category')

ideal_area = pd.DataFrame.from_dict({
    'North-West': 27.0,
    'North-East': 19.6,
    'Center': 20.0,
    'South': 22.6,
    'Islands': 10.8
}, orient='index', columns=['Percentage']).reset_index(names='Category')

ideal_education = pd.DataFrame.from_dict({
    'Non-graduates': 85.0,
    'Graduates': 15.0
}, orient='index', columns=['Percentage']).reset_index(names='Category')

In [ ]:
# ============================================================================
# PLOT APPENDIX A: DEMOGRAPHICS COMPARISON (4-PANEL)
# ============================================================================
# Outputs: demographics_comparison.png, demographics_comparison.pdf
#
# Compares survey sample demographics to Italian national statistics (ISTAT 2025).
# Each panel shows survey data (bars) with national benchmarks (dashed lines).

fig = plt.figure(figsize=(24, 16))
gs = fig.add_gridspec(2, 2, height_ratios=[1, 1], width_ratios=[1, 1], 
                      hspace=0.3, wspace=0.3)

# PANEL 1: Gender (Top Left)
ax1 = fig.add_subplot(gs[0, 0])

# Actual gender stats
gender_counts = df_clean['Q17'].value_counts()
gender_counts = gender_counts[['Donna','Uomo']]
total = gender_counts.sum()
gender_percentages = (gender_counts / total) * 100

# Mapping actual and ideal labels
label_map = {
    "Donna": "Female",
    "Uomo": "Male"
}
gender_percentages = gender_percentages.rename(index=label_map)
gender_percentages = gender_percentages.sort_values(ascending=True)

# Get ideal percentages
ideal_overlay = {
    label_map[k]: ideal_gender.loc[ideal_gender['Category'] == v, 'Percentage'].values[0]
    for k, v in label_map.items()
}

bars = ax1.barh(gender_percentages.index, gender_percentages.values, 
               color=c1, height=0.5)  
overlay_color = c2
legend_added = False  

for idx, category in enumerate(gender_percentages.index):
    if category in ideal_overlay:
        ideal_val = ideal_overlay[category]
        ax1.vlines(x=ideal_val, ymin=idx - 0.3, ymax=idx + 0.3, 
                 colors=overlay_color, linestyles="dashed", linewidth=3,
                 label="National Sample" if not legend_added else None)
        ax1.text(ideal_val + 1, idx, f"{ideal_val:.1f}%", 
               color=overlay_color, va="center", fontsize=20, weight='bold')
        legend_added = True

ax1.grid(False, axis="y")
ax1.set_xlabel("Gender proportion (%)", weight='bold', fontsize=18)
ax1.set_title('a) Gender Distribution', fontsize=20, fontweight='bold')
ax1.set_xlim(0, 60)
ax1.tick_params(axis='both', which='major', labelsize=16)

# PANEL 2: Age (Top Right)
ax2 = fig.add_subplot(gs[0, 1])

# Actual age counts and percentages
age_counts = df_clean['Q6'].value_counts()
total = age_counts.sum()
age_percentages = (age_counts / total) * 100

# Map actual Italian age groups to ideal_age categories for overlay alignment
age_label_map = {
    '18-24 anni': '18–24',
    '25-34 anni': '25–34',
    '35-44 anni': '35–44',
    '45-54 anni': '45–54',
    '55-64 anni': '55–64',
    'Dai 65 anni in su': '65+'}

# Reorder actual data to match ideal age order
ordered_labels = ['18-24 anni', '25-34 anni', '35-44 anni', '45-54 anni', '55-64 anni', 'Dai 65 anni in su']
age_percentages = age_percentages.reindex(ordered_labels)
age_percentages = age_percentages.rename(index=age_label_map)

# Get ideal percentages aligned with mapped labels
ideal_values = ideal_age.set_index('Category').loc[age_percentages.index, 'Percentage'].values

bars = ax2.barh(age_percentages.index, age_percentages.values, 
               color=c1, height=0.5)
overlay_color = c2
legend_added = False

for idx, category in enumerate(age_percentages.index):
    ideal_val = ideal_values[idx]
    ax2.vlines(x=ideal_val, ymin=idx - 0.3, ymax=idx + 0.3, 
             colors=overlay_color, linestyles="dashed", linewidth=3,
             label="National Sample" if not legend_added else None)
    ax2.text(ideal_val + 1, idx, f"{ideal_val:.1f}%", 
           color=overlay_color, va="center", fontsize=20, weight='bold')
    legend_added = True

ax2.grid(False, axis="y")
ax2.set_xlabel("Age proportion (%)", weight='bold', fontsize=18)
ax2.set_title('b) Age Distribution', fontsize=20, fontweight='bold')
ax2.set_xlim(0, max(age_percentages.max(), ideal_values.max()) + 10)
ax2.tick_params(axis='both', which='major', labelsize=16)

# PANEL 3: Area (Bottom Left)
ax3 = fig.add_subplot(gs[1, 0])

# Define the mapping from Italian labels to ideal categories
area_label_map = {
    'Nord-Ovest, Italia': 'North-West',
    'Nord-Est, Italia': 'North-East',
    'Centro Italia': 'Center',
    'Sud Italia': 'South',
    'Isole, Italia': 'Islands'
}

# Define ordered list of Italian labels
ordered_area_labels = list(area_label_map.keys())

# Get value counts for Q10
area_counts = df_clean['Q10'].value_counts()

# Keep only Italian respondents (exclude "Non vivo in Italia")
area_counts_italy = area_counts.loc[ordered_area_labels]
total_italy = area_counts_italy.sum()

# Recalculate actual percentages only over Italian regions
area_percentages = (area_counts_italy / total_italy) * 100
area_percentages = area_percentages.reindex(ordered_area_labels).fillna(0)
area_percentages = area_percentages.rename(index=area_label_map)

# Get ideal values aligned with mapped labels
ideal_values = ideal_area.set_index('Category').loc[area_percentages.index, 'Percentage'].values

bars = ax3.barh(area_percentages.index, area_percentages.values, 
               color=c1, height=0.5)
overlay_color = c2
legend_added = False

for idx, category in enumerate(area_percentages.index):
    ideal_val = ideal_values[idx]
    ax3.vlines(x=ideal_val, ymin=idx - 0.3, ymax=idx + 0.3, 
             colors=overlay_color, linestyles="dashed", linewidth=3,
             label="National Sample" if not legend_added else None)
    ax3.text(ideal_val + 1, idx, f"{ideal_val:.1f}%", 
           color=overlay_color, va="center", fontsize=20, weight='bold')
    legend_added = True

ax3.grid(False, axis="y")
ax3.set_xlabel("Area proportion (%)", weight='bold', fontsize=18)
ax3.set_title('c) Geographic Distribution', fontsize=20, fontweight='bold')
ax3.set_xlim(0, max(area_percentages.max(), ideal_values.max()) + 10)
ax3.tick_params(axis='both', which='major', labelsize=16)

# PANEL 4: Education (Bottom Right)
ax4 = fig.add_subplot(gs[1, 1])

# education mapping
education_map = {k: 'Graduates' for k in [
    'Laurea magistrale o master di primo livello',
    'Laurea triennale o a ciclo unico',
    'Dottorato di ricerca',
    'Master di secondo livello'
]}
education_map.update({k: 'Non-graduates' for k in [
    'Diploma di scuola superiore',
    'Istruzione secondaria di primo grado (medie)'
]})

# Map and calculate percentages
education_simple = df_clean['Istruzione'].map(education_map)
education_simple_counts = education_simple.value_counts()
education_simple_percentages = (education_simple_counts / education_simple_counts.sum()) * 100

# Order to match ideal
ordered_labels = ideal_education['Category'].tolist()
education_simple_percentages = education_simple_percentages.reindex(ordered_labels).fillna(0)
ideal_values = ideal_education.set_index('Category')['Percentage'].values

bars = ax4.barh(education_simple_percentages.index, education_simple_percentages.values, 
               color=c1, height=0.5)
overlay_color = c2
legend_added = False

for idx, category in enumerate(education_simple_percentages.index):
    ideal_val = ideal_values[idx]
    ax4.vlines(x=ideal_val, ymin=idx - 0.3, ymax=idx + 0.3, 
             colors=overlay_color, linestyles="dashed", linewidth=3,
             label="National Sample" if not legend_added else None)
    ax4.text(ideal_val + 1, idx, f"{ideal_val:.1f}%", 
           color=overlay_color, va="center", fontsize=20, weight='bold')
    legend_added = True

ax4.grid(False, axis="y")
ax4.set_xlabel("Education proportion (%)", weight='bold', fontsize=18)
ax4.set_title('d) Education Distribution', fontsize=20, fontweight='bold')
ax4.set_xlim(0, max(education_simple_percentages.max(), ideal_values.max()) + 10)
ax4.tick_params(axis='both', which='major', labelsize=16)

plt.suptitle('Sample Demographics vs National Statistics', fontsize=24, fontweight='bold', y=0.95)
fig.legend(['National Sample'], loc='center', fontsize=18)
plt.subplots_adjust(bottom=0.1)
plt.savefig('../figures/demographics_comparison.png', dpi=300, bbox_inches='tight')
plt.savefig('../figures/demographics_comparison.pdf', dpi=300, bbox_inches='tight')

plt.show()

## Appendix: Education/Training on AI/GenAI

Cross-tabulation of chatbot usage by AI/GenAI training received.

In [ ]:
# ============================================================================
# PLOT APPENDIX B: AI/GENAI EDUCATION/TRAINING
# ============================================================================
# Outputs: lt_course.png, lt_course.pdf
#
# Cross-tabulation showing chatbot usage by AI/GenAI training received.

# Map ALL Italian labels to English
education_mapping = {
    'No': 'None',
    'No, ma ho seguito corsi su altre tecnologie di Intelligenza Artificiale': 'Other AI courses',
    'Sì, cercati in autonomia': 'Self-training',
    'Sì, sul posto di lavoro/a scuola': 'Training at Work/School',
    'Sì, sul posto di lavoro/a scuola,Sì, cercati in autonomia': 'Work/School + Self-training'
}

# Create chatbot usage mapping
df_clean['chatbot_used_en'] = df_clean['chatbot_user'].map({
    'Yes': 'User',
    'No': 'Non-user'
})

# Map education to English
df_clean['education_technology_en'] = df_clean['education_technology'].map(education_mapping)

# Check for unmapped values
unmapped = df_clean[df_clean['education_technology_en'].isna()]['education_technology'].unique()
if len(unmapped) > 0:
    print(f"WARNING: Unmapped education values: {unmapped}")

# Create cross-tabulation
cross = pd.crosstab(df_clean['chatbot_used_en'], df_clean['education_technology_en'], margins=True)

# Calculate percentages
cross_pct = cross.div(cross['All'], axis=0) * 100

# Remove totals for plotting
cross_pct = cross_pct.drop(columns='All').drop(index='All')

# Create figure
fig, ax = plt.subplots(figsize=(8, 4))

# Use distinct colors
colors = ['#1B4F72', '#2E86AB', '#5BA3C5', '#8DC4DC', '#BFE0EE']

# Create stacked bar plot
bars = cross_pct.plot(kind='bar', stacked=True, ax=ax, 
              color=colors, alpha=0.9, edgecolor='white', linewidth=2)

ax.set_facecolor('white')
fig.patch.set_facecolor('white')

# Add percentage labels on bars
for container in ax.containers:
    labels = [f'{v:.1f}%' if v > 5 else '' for v in container.datavalues]
    ax.bar_label(container, labels=labels, label_type='center', 
                fontsize=9, fontweight='bold', color='white')

ax.set_ylabel('Percentage (%)', fontsize=11)
ax.set_xlabel('', fontsize=11)
ax.set_title('Chatbot Use by AI/GenAI Training', fontsize=13, fontweight='bold', pad=15)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0, fontsize=11)
ax.set_ylim(0, 100)

ax.grid(axis='y', alpha=0.3, linestyle='-', linewidth=0.5, color='lightgray')
ax.set_axisbelow(True)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax.legend(title='LT Training', 
         bbox_to_anchor=(1.01, 1), 
         loc='upper left',
         frameon=True,
         fontsize=9,
         title_fontsize=10)

plt.tight_layout()
plt.savefig('../figures/lt_course.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig('../figures/lt_course.pdf', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

## Appendix: Activity Distribution by Demographics

Distribution of activities per user, segmented by age group and gender.

In [ ]:
# ============================================================================
# PLOT APPENDIX C: ACTIVITY DISTRIBUTION BY DEMOGRAPHICS
# ============================================================================
# Outputs: genai_activities_demographics.png, genai_activities_demographics.pdf
#
# Shows distribution of number of activities per user, segmented by:
# - Left panel: Age groups
# - Right panel: Gender (Man/Woman only)

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
fig.suptitle('Distribution of Activities per User by Demographics', fontsize=20, fontweight='bold')

colors = ['#2E86AB', '#A23B72', '#95a5a6']
start_color = '#8DC4DC'
end_color = '#1B4F72'
custom_cmap = LinearSegmentedColormap.from_list("custom_blue", [start_color, end_color])

# Panel C BY AGE GROUP (Left)
ax_c_age = axes[0]

if 'Q52' in df_clean.columns and 'AgeGroup' in df_clean.columns:
    age_groups_sorted = sorted(df_clean['AgeGroup'].dropna().unique())
    n_age_groups = len(age_groups_sorted)
    colors_age = custom_cmap(np.linspace(0, 1, n_age_groups))
    
    for idx, age in enumerate(age_groups_sorted):
        age_subset = df_clean[df_clean['AgeGroup'] == age]
        
        activity_counts = (
            age_subset['Q52']
            .dropna()
            .str.split(',')
            .apply(lambda x: len([i.strip() for i in x if i.strip() not in ['Altro', 'Nessuna di queste']]))
        )
        
        if len(activity_counts) > 0:
            hist_data = activity_counts.value_counts().sort_index()
            avg_activities = activity_counts.mean()
            n_users = len(activity_counts)
            
            ax_c_age.plot(hist_data.index, hist_data.values, 
                         marker='o', linewidth=2.5, markersize=10,
                         label=f'{age} (avg: {avg_activities:.1f}, n={n_users})',
                         color=colors_age[idx], alpha=0.8)
    
    ax_c_age.set_xlabel('Number of Activities', fontsize=16, fontweight='bold')
    ax_c_age.set_ylabel('Number of Users', fontsize=16, fontweight='bold')
    ax_c_age.set_title('Distribution of Activities by Age Group', fontsize=16, fontweight='bold')
    ax_c_age.legend(loc='upper right', fontsize=16)
    ax_c_age.grid(True, alpha=0.2, linestyle='--')

# Panel C BY GENDER (Right)
ax_c_gender = axes[1]

if 'Q52' in df_clean.columns and 'GenderGroup' in df_clean.columns:
    # Filter for only Man and Woman
    df_gender_filtered = df_clean[df_clean['GenderGroup'].isin(['Man', 'Woman'])]
    
    gender_groups_sorted = sorted(df_gender_filtered['GenderGroup'].dropna().unique())
    n_genders = len(gender_groups_sorted)
    colors_gender = custom_cmap(np.linspace(0, 1, n_genders))
    
    for idx, gender in enumerate(gender_groups_sorted):
        gender_subset = df_gender_filtered[df_gender_filtered['GenderGroup'] == gender]
        
        activity_counts = (
            gender_subset['Q52']
            .dropna()
            .str.split(',')
            .apply(lambda x: len([i.strip() for i in x if i.strip() not in ['Altro', 'Nessuna di queste']]))
        )
        
        if len(activity_counts) > 0:
            hist_data = activity_counts.value_counts().sort_index()
            avg_activities = activity_counts.mean()
            n_users = len(activity_counts)
            
            ax_c_gender.plot(hist_data.index, hist_data.values, 
                           marker='o', linewidth=2.5, markersize=10,
                           label=f'{gender} (avg: {avg_activities:.1f}, n={n_users})',
                           color=colors_gender[idx], alpha=0.8)
    
    ax_c_gender.set_xlabel('Number of Activities', fontsize=16, fontweight='bold')
    ax_c_gender.set_ylabel('Number of Users', fontsize=16, fontweight='bold')
    ax_c_gender.set_title('Distribution of Activities by Gender', fontsize=16, fontweight='bold')
    ax_c_gender.legend(loc='upper right', fontsize=16)
    ax_c_gender.grid(True, alpha=0.2, linestyle='--')

plt.tight_layout()
plt.savefig('../figures/genai_activities_demographics.png', dpi=300, bbox_inches='tight')
plt.savefig('../figures/genai_activities_demographics.pdf', dpi=300, bbox_inches='tight')
plt.show()

# 4. Extra Analysis

Additional tables and overall outputs.

In [ ]:
# Display language used breakdown by demographics

# By AgeGroup
print("=== Language Used by Age Group ===\n")
for age_group in sorted(df_clean['AgeGroup'].dropna().unique()):
    print(f"\n{age_group}:")
    df_age = df_clean[df_clean['AgeGroup'] == age_group]
    lang_counts = df_age['language_used'].dropna().str.split(',').explode().str.strip().value_counts()
    total = df_age['language_used'].dropna().shape[0]
    for lang_type, count in lang_counts.items():
        percent = (count / total) * 100
        print(f"  {lang_type}: {count} ({percent:.1f}%)")

# By EducationGroup
print("\n\n=== Language Used by Education Group ===\n")
for edu_group in sorted(df_clean['EducationGroup'].dropna().unique()):
    print(f"\n{edu_group}:")
    df_edu = df_clean[df_clean['EducationGroup'] == edu_group]
    lang_counts = df_edu['language_used'].dropna().str.split(',').explode().str.strip().value_counts()
    total = df_edu['language_used'].dropna().shape[0]
    for lang_type, count in lang_counts.items():
        percent = (count / total) * 100
        print(f"  {lang_type}: {count} ({percent:.1f}%)")

In [ ]:
# Calculate and display LT literacy statistics by age group for chatbot users

# Define literacy items
all_items = {
    'knowledge': 'Good knowledge of LT',
    'prepared': 'Feel prepared to use LT',
    'limitations': 'Know limitations of LT',
    'potential': 'Know potential of LT',
    'recognize_errors': 'Can recognize errors in LT',
    'distinguishai': 'Can distinguish AI from human text',
    'Education': 'Education on LT should be part of programs',
    'Bias_awareness': 'Aware LT can reproduce biases'
}

# Get age groups
age_groups = sorted(df_clean['AgeGroup'].dropna().unique())

# Collect results for chatbot users
users_results = []

for age_group in age_groups:
    df_age = df_clean[df_clean['AgeGroup'] == age_group]
    df_users = df_age[df_age['chatbot_user'] == 'Yes']
    
    for item_key, item_label in all_items.items():
        data = df_users[item_key].dropna()
        mean = data.mean() if len(data) > 0 else np.nan
        
        users_results.append({
            'Age Group': str(age_group),
            'Item': item_label,
            'Mean': mean,
            'n': len(df_users)
        })

# Create summary table
df_users_table = pd.DataFrame(users_results)
pivot_users = df_users_table.pivot(index='Item', columns='Age Group', values='Mean')

# Display results
print("\n" + "="*80)
print("CHATBOT USERS - LITERACY BY AGE GROUP")
print("="*80)
print(pivot_users.round(2).to_string())

# Add average across all items
print("\n" + "-"*80)
print("AVERAGE ACROSS ALL ITEMS:")
avg_row = pivot_users.mean(axis=0)
for age_group in pivot_users.columns:
    print(f"  {age_group}: {avg_row[age_group]:.2f}")
print("="*80)

In [ ]:
# Generate LaTeX table for the literacy results

# Create LaTeX table
latex_output = []
latex_output.append("\\begin{table}[htbp]")
latex_output.append("\\centering")
latex_output.append("\\caption{Language Technology Literacy by Age Group: Chatbot Users}")
latex_output.append("\\label{tab:users_by_age}")
latex_output.append("\\begin{tabular}{l" + "c" * len(age_groups) + "}")
latex_output.append("\\toprule")

# Header row
header = "\\textbf{Item} & " + " & ".join([f"\\textbf{{{age}}}" for age in age_groups]) + " \\\\"
latex_output.append(header)
latex_output.append("\\midrule")

# Data rows
for item_label in all_items.values():
    row_data = [item_label]
    for age_group in age_groups:
        # Get mean for this item and age group
        mask = (df_users_table['Item'] == item_label) & (df_users_table['Age Group'] == str(age_group))
        mean_val = df_users_table[mask]['Mean'].values[0]
        row_data.append(f"{mean_val:.2f}")
    
    row = " & ".join(row_data) + " \\\\"
    latex_output.append(row)

# Average row
latex_output.append("\\midrule")
avg_data = ["\\textbf{Average across all items}"]
for age_group in age_groups:
    avg_val = pivot_users[str(age_group)].mean()
    avg_data.append(f"\\textbf{{{avg_val:.2f}}}")
row = " & ".join(avg_data) + " \\\\"
latex_output.append(row)

latex_output.append("\\bottomrule")
latex_output.append("\\end{tabular}")

# Add note with sample sizes
note = "\\begin{tablenotes}[para,flushleft]\n\\small\n\\textit{Note.} Sample sizes: "
sample_sizes = []
for age_group in age_groups:
    df_age = df_clean[df_clean['AgeGroup'] == age_group]
    df_users = df_age[df_age['chatbot_user'] == 'Yes']
    sample_sizes.append(f"{age_group} ($n$={len(df_users)})")
note += ", ".join(sample_sizes) + ".\n\\end{tablenotes}"
latex_output.append(note)

latex_output.append("\\end{table}")

# Print LaTeX code
latex_code = "\n".join(latex_output)
print("\n" + "="*80)
print("LATEX TABLE - CHATBOT USERS")
print("="*80 + "\n")
print(latex_code)

# Save to file
with open('../figures/users_by_age_latex.tex', 'w') as f:
    f.write(latex_code)

# 5. Statistical Modeling Figures

Figures based on regression results from `4_modeling.ipynb`. Results are loaded from saved TSV files in `../results/`.

## 5.1 GenAI Adoption - Odds Ratios Forest Plot

In [ ]:
# Load logistic regression results
RESULTS_DIR = PROJECT_ROOT / "results"
logit_r_adoption = pd.read_csv(RESULTS_DIR / "logit_r_adoption.tsv", sep='\t')
print(f"✓ Loaded odds ratios data: {logit_r_adoption.shape}")
logit_r_adoption.head()

In [ ]:
# Create forest plot
fig, ax = plt.subplots(figsize=(8, 5))

# Create simple evenly spaced y positions 
n_vars = len(logit_r_adoption)
y_pos = list(range(n_vars-1, -1, -1)) 

# Add alternating background bands by category
categories = logit_r_adoption['category'].tolist()
unique_categories = []
for cat in categories:
    if cat not in unique_categories:
        unique_categories.append(cat)

# Create bands for each category group
band_color = True  
for cat in unique_categories:
    # Find all indices for this category
    cat_indices = [i for i, row_cat in enumerate(categories) if row_cat == cat]
    
    if cat_indices:
        # Get y positions for this category
        cat_y_positions = [y_pos[i] for i in cat_indices]
        y_min = min(cat_y_positions) - 0.5
        y_max = max(cat_y_positions) + 0.5
        
        # Add band
        if band_color:  
            ax.axhspan(y_min, y_max, alpha=0.1, color='gray', zorder=0)        
        band_color = not band_color  

# Define a single color for all points and lines
main_color = tri_palette_colors[0]

# Plot odds ratios and confidence intervals
for i, (idx, row) in enumerate(logit_r_adoption.iterrows()):
    # Plot CI line (always solid, same color)
    ax.plot([row['ci_lower'], row['ci_upper']], [y_pos[i], y_pos[i]],
            color=main_color, linewidth=1.5, alpha=0.8)
    
    # Plot point estimate
    if row['significant']:
        # Filled circle for significant
        ax.scatter(row['odds_ratio'], y_pos[i],
                   color=main_color, edgecolor=main_color,
                   marker='o', s=80, linewidth=1.5)
    else:
        # Hollow circle for non-significant
        ax.scatter(row['odds_ratio'], y_pos[i],
                   facecolors='none', edgecolor=main_color,
                   marker='o', s=80, linewidth=1.5)

# Add vertical line at OR = 1
ax.axvline(x=1, color=tri_palette_colors[1], linestyle='--', alpha=0.7, linewidth=1)

# Axis and Titles
ax.set_yticks(y_pos)
ax.set_yticklabels([f"{row['variable']}" for idx, row in logit_r_adoption.iterrows()])
ax.set_xlabel('OR (95% CI)', fontsize=12, fontweight='bold')
ax.set_title('Odds Ratios for GenAI Adoption', 
             fontsize=14, fontweight='bold')

# Calculate symmetric limits around 1 on log scale
log_ors = np.log10(logit_r_adoption['odds_ratio'])
log_cis = np.log10(np.concatenate([logit_r_adoption['ci_lower'], logit_r_adoption['ci_upper']]))
max_log_dist = max(abs(log_cis).max(), abs(log_ors).max()) * 1.1  # Add 10% padding

# Set symmetric limits around 1 on log scale
x_min = 10**(-max_log_dist)
x_max = 10**(max_log_dist)

# Set x-axis to log scale for better visualization
ax.set_xscale('log')
ax.set_xlim(x_min, x_max)

# Add custom x-axis labels showing key values
ax.set_xticks([0.1, 1, 10, 50])
ax.set_xticklabels(['0.1', '1', '10', '50'])

# Add grid lines: major in bold (optional), minor as light lines
ax.grid(which='minor', axis='x', linestyle='--', color='lightgray', alpha=0.2)
ax.grid(which='major', axis='y', linestyle='-', color='lightgray', alpha=0.2)

# Add category labels on the right (aligned with first variable of each category)
seen_categories = set()
for i, (idx, row) in enumerate(logit_r_adoption.iterrows()):
    if row['category'] not in seen_categories:
        ax.text(ax.get_xlim()[1] * 1.1, y_pos[i], row['category'],  color='gray', style='italic', ha='left', va='center', fontsize=10)
        seen_categories.add(row['category'])

# Hide top and right spines
ax.spines['left'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)

ax.margins(y=0)

plt.tight_layout()
# plt.savefig(FIGURES_DIR / 'or_genai.png', dpi=300, bbox_inches='tight')
# plt.savefig(FIGURES_DIR / 'or_genai.pdf', dpi=300, bbox_inches='tight')
plt.show()

## 5.2 OLS Intent Frequency - Regression Coefficients

In [ ]:
# Load OLS results for all intents from separate files
intent_files = {
    'Information Retrieval': 'ols_information_retrieval_summary.tsv',
    'Problem Solving': 'ols_problem_solving_summary.tsv',
    'Learning': 'ols_learning_summary.tsv',
    'Content Creation': 'ols_content_creation_summary.tsv',
    'Leisure': 'ols_leisure_summary.tsv',
    'Creativity': 'ols_creativity_summary.tsv'
}

# Variable name mapping (from model output to display names)
var_name_map = {
    'Intercept': 'Constant',
    'C(GenderGroup)[T.Woman]': 'Woman',
    'C(AgeGroup)[T.35-54]': '35-54',
    'C(AgeGroup)[T.55-64]': '55-64',
    'C(AgeGroup)[T.65+]': '65+',
    'C(GeographyGroup)[T.Centre]': 'Centre',
    'C(GeographyGroup)[T.South and Islands]': 'South&Islands',
    'C(GeographyGroup)[T.Abroad]': 'Abroad',
    'C(IncomeGroup)[T.Mid]': 'Mid',
    'C(IncomeGroup)[T.Higher]': 'Higher',
    'C(EducationGroup)[T.Graduates]': 'Graduates',
    'LT_exp': 'LT experience',
    'lt_lit01': 'LT literacy'
}

# Prepare dataframe for plotting
data = {
    'model': [],
    'variable': [],
    'coefficient': [],
    'std_error': [],
    'ci_lower': [],
    'ci_upper': [],
    'significance': []
}

# Load and combine all files
for intent_name, filename in intent_files.items():
    file_path = RESULTS_DIR / filename
    df_intent = pd.read_csv(file_path, sep='\t', index_col=0)
    
    for var_name, row in df_intent.iterrows():
        # Map variable name
        display_name = var_name_map.get(var_name, var_name)
        
        data['model'].append(intent_name)
        data['variable'].append(display_name)
        data['coefficient'].append(row['Coef.'])
        data['std_error'].append(row['Std.Err.'])
        data['ci_lower'].append(row['[0.025'])
        data['ci_upper'].append(row['0.975]'])
        
        # Add significance stars
        p_value = row['P>|t|']
        if p_value < 0.001:
            data['significance'].append('***')
        elif p_value < 0.01:
            data['significance'].append('**')
        elif p_value < 0.05:
            data['significance'].append('*')
        else:
            data['significance'].append('')

df_usage_ols = pd.DataFrame(data)
print(f"✓ Loaded OLS usage data from {len(intent_files)} files: {df_usage_ols.shape}")

# Baseline mapping
baseline_mapping = {
    'Woman': 'Gender (Man)',
    '35-54': 'Age (18-34)',
    '55-64': 'Age (18-34)',
    '65+': 'Age (18-34)',
    'Graduates': 'Education (Non-graduates)',
    'Mid': 'Income (Lower)',
    'Higher': 'Income (Lower)',
    'Centre': 'Geography (North)',
    'South&Islands': 'Geography (North)',
    'Abroad': 'Geography (North)',
    'LT experience': 'LT experience',
    'LT literacy': 'LT literacy'
}

df_usage_ols['baseline'] = df_usage_ols['variable'].map(baseline_mapping)
print(f"✓ Prepared {len(df_usage_ols)} coefficient estimates for plotting")

In [ ]:
# Remove the Constant row, geography/income variables, Woman, AND Graduates
df_plot = df_usage_ols[
    (df_usage_ols['variable'] != 'Constant') &
    (df_usage_ols['variable'] != 'Woman') &
    (df_usage_ols['variable'] != 'Graduates') &
    (~df_usage_ols['variable'].isin(['Mid', 'Higher', 'Centre', 'South&Islands', 'Abroad']))
].copy()

# Define colors for each model
model_colors = {
    'Information Retrieval': tri_palette_colors[0],
    'Problem Solving': tri_palette_colors[1],
    'Learning': '#2ecc71',
    'Content Creation': '#f39c12',
    'Leisure': '#9b59b6',
    'Creativity': blues_palette[3]
}

# Marker styles of significance
marker_style = 'o'  

# Get unique variables and baselines (in order of first appearance)
variables = df_plot.drop_duplicates('variable')['variable'].tolist()
baselines = [df_plot[df_plot['variable'] == v]['baseline'].iloc[0] for v in variables]

# Create figure with more horizontal space 
fig, ax = plt.subplots(figsize=(12, 8))

# Position parameters 
x_positions = np.arange(len(variables)) * 1.5
n_models = len(model_colors)
offset_width = 0.2
offsets = np.linspace(-offset_width * (n_models-1)/2, 
                      offset_width * (n_models-1)/2, 
                      n_models)

# Add alternating background bands by baseline category
baseline_groups = []
current_baseline = baselines[0]
start_idx = 0

for i in range(len(baselines)):
    if i == len(baselines) - 1:
        # Last item
        if baselines[i] == current_baseline:
            baseline_groups.append((start_idx, i + 1, current_baseline))
        else:
            baseline_groups.append((start_idx, i, current_baseline))
            baseline_groups.append((i, i + 1, baselines[i]))
    elif baselines[i] != current_baseline:
        baseline_groups.append((start_idx, i, current_baseline))
        start_idx = i
        current_baseline = baselines[i]

# Draw alternating bands
for idx, (start, end, baseline) in enumerate(baseline_groups):
    if idx % 2 == 0:  # Alternate gray bands
        ax.axvspan(x_positions[start] - 0.75, 
                   x_positions[end - 1] + 0.75, 
                   facecolor='lightgray', alpha=0.2, zorder=0)

# Plot each model
for i, (model, color) in enumerate(model_colors.items()):
    model_data = df_plot[df_plot['model'] == model].copy()
    
    x_pos = x_positions + offsets[i]
    
    # Create a mapping from variable to data
    var_to_data = {row['variable']: row for _, row in model_data.iterrows()}
    
    # Plot confidence intervals and coefficients for each variable
    for j, var in enumerate(variables):
        if var in var_to_data:
            row = var_to_data[var]
            if pd.notna(row['coefficient']):
                sig_level = row['significance']
                is_significant = (sig_level in ['*', '**', '***'])
                
                # Determine line style
                linestyle = '-' if is_significant else '--'
                linewidth = 2 if is_significant else 1
                alpha = 0.8 if is_significant else 0.3
                
                # Plot confidence interval
                ax.plot([x_pos[j], x_pos[j]],
                       [row['ci_lower'], row['ci_upper']], 
                       color=color, linewidth=linewidth, 
                       linestyle=linestyle, alpha=alpha)
                
                # Determine marker fill (simplified - just filled or empty)
                if is_significant:
                    facecolor = color
                else:
                    facecolor = 'white'
                
                edgecolor = color
                edgewidth = 1.5
                
                # Plot coefficient
                ax.scatter(x_pos[j], row['coefficient'], 
                          marker=marker_style,
                          s=100,
                          facecolors=facecolor,
                          edgecolors=edgecolor,
                          linewidth=edgewidth,
                          zorder=3)

# Add reference line at 0 
ax.axhline(y=0, color='black', linestyle='-', linewidth=1.5, alpha=0.7)

# Customize plot
ax.set_xticks(x_positions)
ax.set_xticklabels(variables, fontsize=11, rotation=45, ha='right')
ax.set_ylabel('Coefficient', fontsize=12, fontweight='bold')
ax.set_title('Regression Coefficients for Usage Frequency across Intents', 
             fontsize=14, fontweight='bold', pad=20)

# Add baseline info at top of plot
ax3 = ax.twiny()
ax3.set_xticks(x_positions)
ax3.set_xticklabels([f'{b}' for b in baselines], fontsize=9, color='gray', style='italic', rotation=45, ha='left')
ax3.set_xlim(ax.get_xlim())

# Create custom legend
from matplotlib.lines import Line2D

# Model colors
legend_models = [Line2D([0], [0], marker='o', color='w', 
                        markerfacecolor=color, markersize=8, label=model,
                        markeredgecolor=color, markeredgewidth=1.5)
                for model, color in model_colors.items()]

# Add legends
legend1 = ax.legend(handles=legend_models, loc='upper left', frameon=True, fontsize=10, title_fontsize=9)

# Grid (now horizontal)
ax.grid(axis='y', alpha=0.3, linestyle=':')
ax.set_axisbelow(True)

plt.tight_layout()
# Save figure
# plt.savefig(FIGURES_DIR / 'intent_predictor.png', dpi=300, bbox_inches='tight')
# plt.savefig(FIGURES_DIR / 'intent_predictor.pdf', bbox_inches='tight')
plt.show()

## 5.3 Gender Gaps - Adoption by Age & Usage by Intent

In [ ]:
# Load AME results for adoption (by age) and usage (by intent)
ame_adoption_df = pd.read_csv(RESULTS_DIR / "logistic_adoption_ame_by_age.tsv", sep='\t')
ame_usage_df = pd.read_csv(RESULTS_DIR / "ols_usage_intensity_ame_by_intent.tsv", sep='\t')

print(f"✓ Loaded adoption AME data: {ame_adoption_df.shape}")
print(f"✓ Loaded usage AME data: {ame_usage_df.shape}")

ame_adoption_df.head()

In [ ]:
# Create comparison plots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5), facecolor='white')

# ============================================================================
# LEFT PANEL (A): Adoption by age - AME version
# ============================================================================

age_groups = ame_adoption_df['age_group'].tolist()
gender_gaps_ame = ame_adoption_df['ame'].tolist()
ci_lower_ame = ame_adoption_df['ci_lower'].tolist()
ci_upper_ame = ame_adoption_df['ci_upper'].tolist()

x_pos_age = np.arange(len(age_groups))

# Plot confidence intervals
for i in range(len(age_groups)):
    ax1.plot([i, i], [ci_lower_ame[i], ci_upper_ame[i]], color=tri_palette_colors[0], linewidth=2.5, alpha=0.6)

# Plot line and points
ax1.plot(x_pos_age, gender_gaps_ame, 
        marker='o', color=tri_palette_colors[0], linewidth=2, markersize=12,
        markerfacecolor=tri_palette_colors[0], markeredgecolor='white')

# Add zero reference line
ax1.axhline(y=0, color=tri_palette_colors[1], linestyle='--', linewidth=2, alpha=0.7)

# Formatting
ax1.set_xticks(x_pos_age)
ax1.set_xticklabels(age_groups, fontsize=15, rotation=45)
ax1.set_xlabel('', fontsize=14, fontweight='bold')
ax1.set_ylabel('Difference in Probability\n(Woman - Man)', fontsize=15, fontweight='bold')
ax1.set_title('a) Gender Gap in Adoption by Age Group', fontsize=17, fontweight='bold', pad=20, loc='left')
ax1.grid(axis='y', alpha=0.2, linestyle='-', linewidth=0.5, color='gray')
ax1.set_axisbelow(True)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# ============================================================================
# RIGHT PANEL (B): Usage intensity by intent - AME version
# ============================================================================

intents_sorted = ame_usage_df['intent'].tolist()
gaps_intent = ame_usage_df['gap'].tolist()
ci_lowers_intent = ame_usage_df['ci_lower'].tolist()
ci_uppers_intent = ame_usage_df['ci_upper'].tolist()
pvals_intent = ame_usage_df['pval'].tolist()

x_pos_intent = np.arange(len(intents_sorted))

# Plot confidence intervals
for i in range(len(intents_sorted)):
    ax2.plot([i, i], [ci_lowers_intent[i], ci_uppers_intent[i]], 
            color=tri_palette_colors[0], linewidth=2.5, alpha=0.6)

# Plot points with significance indicated
for i in range(len(intents_sorted)):
    is_significant = pvals_intent[i] < 0.05
    marker_face = tri_palette_colors[0] if is_significant else 'white'
    
    ax2.scatter(i, gaps_intent[i], marker='o', s=70, facecolors=marker_face, edgecolors=tri_palette_colors[0], linewidth=2, zorder=3)

# Connect points with line
ax2.plot(x_pos_intent, gaps_intent, color=tri_palette_colors[0], linewidth=2, alpha=0.8, zorder=2)

# Add zero reference line
ax2.axhline(y=0, color=tri_palette_colors[1], linestyle='--', linewidth=2, alpha=0.7, zorder=0)

# Formatting
ax2.set_xticks(x_pos_intent)
ax2.set_xticklabels(intents_sorted, fontsize=15, rotation=45, ha='right')
ax2.set_xlabel('', fontsize=12, fontweight='bold')
ax2.set_ylabel('', fontsize=12, fontweight='bold')
ax2.set_title('b) Gender Gap in Usage Frequency by Intent', fontsize=17, fontweight='bold', pad=20, loc='left')
ax2.grid(axis='y', alpha=0.2, linestyle='-', linewidth=0.5, color='gray')
ax2.set_axisbelow(True)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

# Overall figure adjustments
plt.tight_layout()

# Save figure
# plt.savefig(FIGURES_DIR / 'gender_gaps_combined.png', dpi=300, bbox_inches='tight')
# plt.savefig(FIGURES_DIR / 'gender_gaps_combined.pdf', bbox_inches='tight')
plt.show()